# SRΨ-v2.0: 真实 Burgers 数据准备

## 🎯 目标

准备真实的 Burgers 方程数据用于 SRΨ-v2.0 训练

数据参数:
- 方程: 1D Burgers
- 雷诺数: R = 10
- 粘度: ν = 0.01
- 样本数: 4800
- 时间步: 48
- 空间点: 128

---

## Step 1: 克隆仓库并检查数据

In [ ]:
# Clone repository
!git clone https://github.com/Mozilla2004/srpsi-engine-tiny.git
%cd srpsi-engine-tiny
!git pull

print("\n" + "="*70)
print(" " * 20 + "REPOSITORY READY")
print("="*70)

## Step 2: 加载并验证数据

In [ ]:
import numpy as np
import torch
from pathlib import Path

print("\n" + "="*70)
print(" " * 25 + "DATA VALIDATION")
print("="*70)

# Load data
data_path = 'data/burgers_1d.npy'

if Path(data_path).exists():
    print(f"\n📁 Loading data from: {data_path}")
    u_data = np.load(data_path)
    
    print(f"\n✅ Data loaded successfully!")
    print(f"\n📊 Data Statistics:")
    print(f"   Shape: {u_data.shape}")
    print(f"   Dtype: {u_data.dtype}")
    print(f"   Size: {u_data.nbytes / 1024 / 1024:.1f} MB")
    print(f"\n   Value Range: [{u_data.min():.6f}, {u_data.max():.6f}]")
    print(f"   Mean: {u_data.mean():.6f}")
    print(f"   Std Dev: {u_data.std():.6f}")
    
    # Check first few samples
    print(f"\n🔍 Sample Inspection (first 3 samples):")
    for i in range(3):
        sample = u_data[i]
        print(f"   Sample {i}:")
        print(f"     Shape: {sample.shape}")
        print(f"     Range: [{sample.min():.3f}, {sample.max():.3f}]")
        print(f"     Mean: {sample.mean():.3f}")
    
    print("\n" + "="*70)
    print(" " * 20 + "DATA VALIDATION PASSED ✅")
    print("="*70)
    
else:
    print(f"\n❌ Data file not found: {data_path}")
    print("\n💡 Please ensure burgers_1d.npy is in the data/ directory")

## Step 3: 数据分割 (Train/Val/Test)

In [ ]:
print("\n" + "="*70)
print(" " * 25 + "DATA SPLITTING")
print("="*70)

# Split parameters
num_samples = u_data.shape[0]
train_ratio = 0.8
val_ratio = 0.1
test_ratio = 0.1

num_train = int(num_samples * train_ratio)
num_val = int(num_samples * val_ratio)
num_test = num_samples - num_train - num_val

# Split
u_train = u_data[:num_train]
u_val = u_data[num_train:num_train+num_val]
u_test = u_data[num_train+num_val:]

print(f"\n📊 Split Results:")
print(f"   Total samples: {num_samples}")
print(f"   Train: {num_train} ({train_ratio*100:.0f}%)")
print(f"   Val: {num_val} ({val_ratio*100:.0f}%)")
print(f"   Test: {num_test} ({test_ratio*100:.0f}%)")

print(f"\n📦 Shapes:")
print(f"   Train: {u_train.shape}")
print(f"   Val: {u_val.shape}")
print(f"   Test: {u_test.shape}")

print("\n" + "="*70)
print(" " * 20 + "DATA SPLITTING PASSED ✅")
print("="*70)

## Step 4: 转换为训练格式 (重要!)

In [ ]:
import torch

print("\n" + "="*70)
print(" " * 20 + "DATA TRANSFORMATION")
print("="*70)

# Model parameters
tin = 16   # Input time steps
tout = 32  # Output time steps
batch_size = 100

print(f"\n📝 Model Configuration:")
print(f"   Input steps (tin): {tin}")
print(f"   Output steps (tout): {tout}")
print(f"   Batch size: {batch_size}")

# Prepare training data
# Input: [num_samples, nx, tin] - Transposed!
# Output: [num_samples, nx, tout] - Transposed!

train_x = torch.tensor(u_train[:, :tin, :], dtype=torch.float32).transpose(1, 2)
train_y = torch.tensor(u_train[:, tin:tin+tout, :], dtype=torch.float32).transpose(1, 2)

val_x = torch.tensor(u_val[:, :tin, :], dtype=torch.float32).transpose(1, 2)
val_y = torch.tensor(u_val[:, tin:tin+tout, :], dtype=torch.float32).transpose(1, 2)

test_x = torch.tensor(u_test[:, :tin, :], dtype=torch.float32).transpose(1, 2)
test_y = torch.tensor(u_test[:, tin:tin+tout, :], dtype=torch.float32).transpose(1, 2)

print(f"\n✅ Data transformed!")
print(f"\n📦 Final Shapes:")
print(f"   Train X: {train_x.shape} (should be [{num_train}, 128, {tin}])")
print(f"   Train Y: {train_y.shape} (should be [{num_train}, 128, {tout}])")
print(f"   Val X: {val_x.shape}")
print(f"   Val Y: {val_y.shape}")
print(f"   Test X: {test_x.shape}")
print(f"   Test Y: {test_y.shape}")

# Verify shapes
assert train_x.shape == (num_train, 128, tin), f"Train X shape mismatch!"
assert train_y.shape == (num_train, 128, tout), f"Train Y shape mismatch!"
assert val_x.shape == (num_val, 128, tin), f"Val X shape mismatch!"
assert val_y.shape == (num_val, 128, tout), f"Val Y shape mismatch!"

print("\n✅ All shapes verified!")

print("\n" + "="*70)
print(" " * 20 + "DATA TRANSFORMATION PASSED ✅")
print("="*70)

## Step 5: 保存预处理后的数据

In [ ]:
import os

print("\n" + "="*70)
print(" " * 20 + "SAVING PROCESSED DATA")
print("="*70)

# Create data directory
data_dir = Path('data/processed')
data_dir.mkdir(parents=True, exist_ok=True)

# Save as numpy arrays (for fast loading)
np.save(data_dir / 'train_x.npy', train_x.numpy())
np.save(data_dir / 'train_y.npy', train_y.numpy())
np.save(data_dir / 'val_x.npy', val_x.numpy())
np.save(data_dir / 'val_y.npy', val_y.numpy())
np.save(data_dir / 'test_x.npy', test_x.numpy())
np.save(data_dir / 'test_y.npy', test_y.numpy())

# Save metadata
metadata = {
    'num_samples': num_samples,
    'num_train': num_train,
    'num_val': num_val,
    'num_test': num_test,
    'tin': tin,
    'tout': tout,
    'nx': 128,
    'total_steps': 48,
    'train_x_shape': list(train_x.shape),
    'train_y_shape': list(train_y.shape),
}

import json
with open(data_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"\n✅ Data saved to: {data_dir}")
print(f"\n📁 Saved Files:")
for file in data_dir.glob('*'):
    size_mb = file.stat().st_size / 1024 / 1024
    print(f"   {file.name}: {size_mb:.2f} MB")

print("\n" + "="*70)
print(" " * 15 + "DATA SAVED SUCCESSFULLY ✅")
print("="*70)

## Step 6: 验证加载的数据

In [ ]:
print("\n" + "="*70)
print(" " * 20 + "VERIFYING SAVED DATA")
print("="*70)

# Load back
train_x_loaded = torch.from_numpy(np.load(data_dir / 'train_x.npy'))
train_y_loaded = torch.from_numpy(np.load(data_dir / 'train_y.npy'))

print(f"\n✅ Data loaded back successfully!")
print(f"\n📊 Verification:")
print(f"   Original Train X: {train_x.shape}")
print(f"   Loaded Train X:  {train_x_loaded.shape}")
print(f"   Match: {torch.allclose(train_x, train_x_loaded)}")

# Check metadata
with open(data_dir / 'metadata.json', 'r') as f:
    metadata_loaded = json.load(f)

print(f"\n📋 Metadata:")
for key, value in metadata_loaded.items():
    print(f"   {key}: {value}")

print("\n" + "="*70)
print(" " * 20 + "VERIFICATION PASSED ✅")
print("="*70)

## Step 7: 数据可视化 (可选)

In [ ]:
import matplotlib.pyplot as plt

print("\n" + "="*70)
print(" " * 20 + "DATA VISUALIZATION")
print("="*70)

# Plot first 3 samples from training set
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for i in range(3):
    # Get sample (transpose back for plotting: [nx, tin] -> [tin, nx])
    sample = train_x[i].numpy().transpose(1, 0)
    
    # Plot evolution
    im = axes[i].imshow(sample, aspect='auto', cmap='RdBu_r', 
                      vmin=-2, vmax=2, origin='lower')
    axes[i].set_title(f'Training Sample {i+1}')
    axes[i].set_xlabel('Space (x)')
    axes[i].set_ylabel('Time (t)')
    fig.colorbar(im, ax=axes[i])

plt.tight_layout()
plt.savefig(data_dir / 'sample_visualization.png', dpi=150)
plt.show()

print(f"\n✅ Visualization saved to: {data_dir / 'sample_visualization.png'}")

## Step 8: 创建简化的加载函数

In [ ]:
print("\n" + "="*70)
print(" " * 20 + "CREATING HELPER FUNCTIONS")
print("="*70)

# Create a helper function to load processed data
helper_code = '''
import torch
import numpy as np
from pathlib import Path

def load_processed_data(data_dir='data/processed'):
    """
    Load pre-processed Burgers data.
    
    Returns:
        Dictionary with train_x, train_y, val_x, val_y, test_x, test_y
    """
    data_dir = Path(data_dir)
    
    data = {}
    for split in ['train', 'val', 'test']:
        data[f'{split}_x'] = torch.from_numpy(np.load(data_dir / f'{split}_x.npy'))
        data[f'{split}_y'] = torch.from_numpy(np.load(data_dir / f'{split}_y.npy'))
    
    return data

# Usage:
# data = load_processed_data()
# train_x = data['train_x']  # [num_train, 128, 16]
# train_y = data['train_y']  # [num_train, 128, 32]
'''

# Save helper function
with open('data/load_processed_data.py', 'w') as f:
    f.write(helper_code)

print("\n✅ Helper function saved to: data/load_processed_data.py")
print("\n💡 Usage in training script:")
print("   from data.load_processed_data import load_processed_data")
print("   data = load_processed_data()")
print("   train_x = data['train_x']")

print("\n" + "="*70)
print(" " * 15 + "PREPARATION COMPLETE ✅")
print("="*70)

print("\n🎯 Next Steps:")
print("   1. Upload data/processed/ to Colab")
print("   2. Run SRΨ-v2.0 training with real data")
print("   3. Expect Energy Drift < 0.3, Momentum Drift < 2.0")